In [ ]:
#NAIVE RAG using Llama Index

# %pip install llama-index llama-index-llms-ollama llama-index-embeddings-ollama
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

In [ ]:
#configuration
Settings.llm = Ollama(model="gemma4:e4b", temperature=0)
Settings.embed_model = OllamaEmbedding(model_name="qwen3-embedding:0.6b")
Settings.chunk_size_limit = 512
Settings.chunk_overlap = 50

In [15]:
#document loading

from llama_index.core import SimpleDirectoryReader
documents = SimpleDirectoryReader('train.csv').load_data()

ValueError: Directory train.csv does not exist.

In [16]:
#main

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import VectorStoreIndex

# 1. Parse into nodes explicitly
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)

# 2. Build the index from nodes (not from raw documents)
index = VectorStoreIndex(nodes)

# 3. Build retriever explicitly (this is the piece that changes across architectures)
retriever = index.as_retriever(similarity_top_k=5)

# 4. Build query engine from the retriever
from llama_index.core.query_engine import RetrieverQueryEngine
query_engine = RetrieverQueryEngine.from_args(retriever)

NameError: name 'documents' is not defined

In [ ]:
#Query

response = query_engine.query("What are the main causes of hallucination in RAG systems?")
print(response.response)
print([n.text for n in response.source_nodes])  # this is your `contexts` field for RAGAS/DeepEval